# HIV NSPPK Bayesian optimization

Tune NSPPK encoder parameters with `NSPPKClassifier.tune()`, then refit the selected encoder and fixed random forest on the full training split. The final cell reports full-data held-out performance with a metric chart, confusion matrices, ROC curve, and precision–recall curve.

In [ ]:
from pathlib import Path
import pickle
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

REPO_CANDIDATES = [Path.cwd().resolve(), Path.cwd().resolve().parent]
REPO_ROOT = next(path for path in REPO_CANDIDATES if (path / 'src').exists())
SRC_DIR = REPO_ROOT / 'src'
NOTEBOOK_DIR = REPO_ROOT / 'notebooks'
for path in (SRC_DIR, NOTEBOOK_DIR):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from nsppk import NSPPK, NSPPKClassifier
from hiv_tuning_utils import (
    balanced_training_sample, plot_heldout_report, plot_tuning_progress,
    split_train_test, tuning_result_frame,
)

# User-editable experiment parameters.
RANDOM_STATE = 2026
BUDGET_PER_CLASS = 2_000
TEST_SIZE = 0.25
CV_FOLDS = 3
N_TRIALS = 20
N_INITIAL_POINTS = 10
NBITS = 14
PARALLEL = True
SEARCH_SPACE = {
    'radius': (1, 3),
    'distance': (2, 6),
    'connector': (0, 2),
    'use_simplex_features': [False, True],
    'simplex_feature_order': [2, 3],
}
RF_SETTINGS = {
    'n_estimators': 300,
    'criterion': 'gini',
    'max_features': 'sqrt',
    'max_depth': None,
    'min_samples_split': 2,
    'min_samples_leaf': 1,
    'bootstrap': True,
    'class_weight': 'balanced_subsample',
    'n_jobs': -1,
    'random_state': RANDOM_STATE,
}
HIV_PICKLE = REPO_ROOT / 'data' / 'HIV_networkx_graphs.pkl'
print('BO search space:', SEARCH_SPACE)
print(f'BO budget: {BUDGET_PER_CLASS:,} per class; hash width: {NBITS}')
print('Fixed random forest:', RF_SETTINGS)

## Prepare the full train/test split and BO cohort

The test split is reserved before tuning. The user-defined balanced sample is drawn only from the training partition; its effective size is capped by the available active examples.

In [ ]:
with HIV_PICKLE.open('rb') as handle:
    hiv_data = pickle.load(handle)
all_graphs = hiv_data['graphs']
y_all = np.asarray(hiv_data['labels'], dtype=np.int8)

if not np.array_equal(np.unique(y_all), np.array([0, 1])):
    raise ValueError('Expected the binary HIV activity labels 0 and 1.')
full_train_idx, full_test_idx = split_train_test(
    y_all, test_size=TEST_SIZE, random_state=RANDOM_STATE
)
tuning_idx, effective_per_class = balanced_training_sample(
    y_all, full_train_idx, BUDGET_PER_CLASS, random_state=RANDOM_STATE
)

tuning_graphs = [all_graphs[i] for i in tuning_idx]
y_tuning = y_all[tuning_idx]
full_train_graphs = [all_graphs[i] for i in full_train_idx]
y_full_train = y_all[full_train_idx]
full_test_graphs = [all_graphs[i] for i in full_test_idx]
y_full_test = y_all[full_test_idx]

print(f'Full dataset: {len(all_graphs):,}; classes={np.bincount(y_all).tolist()}')
print(f'Train/test: {len(full_train_idx):,}/{len(full_test_idx):,}')
print(f'Train classes: {np.bincount(y_full_train).tolist()}')
print(f'Test classes: {np.bincount(y_full_test).tolist()}')
print(f'Effective tuning sample: {len(tuning_idx):,} ({effective_per_class:,} per class)')

## Tune the NSPPK encoder

The classifier configuration remains fixed across all trials. `tune()` performs the Latin-hypercube and Gaussian-process search, then refits the winning wrapper on the tuning cohort.

In [ ]:
tuning_model = NSPPKClassifier(
    estimator='rf',
    estimator_params=RF_SETTINGS,
    nbits=NBITS,
    dense=False,
    parallel=PARALLEL,
)
tuning_started = time.perf_counter()
tune_result = tuning_model.tune(
    tuning_graphs,
    y_tuning,
    search_space=SEARCH_SPACE,
    scoring='average_precision',
    cv=CV_FOLDS,
    n_samples=N_TRIALS,
    n_initial_points=N_INITIAL_POINTS,
    random_state=RANDOM_STATE,
    verbose=True,
)
tuning_seconds = time.perf_counter() - tuning_started
best_params = dict(tune_result.best_params)
print(f'BO completed in {tuning_seconds / 60:.1f} minutes')
print('Best NSPPK settings:', best_params)
print(f'Best mean CV average precision: {tune_result.best_score:.4f}')

## Review BO history

In [ ]:
bo_results = tuning_result_frame(tune_result)
display(bo_results.head(10).round(4))
progress_figure = plot_tuning_progress(
    bo_results, score_column='score', title='HIV NSPPK Bayesian optimization progress'
)
plt.show()

## Full-data train/test experiment

Expose `NSPPK` directly here: build the best graph features for the complete training split, transform the untouched test split, and fit the same fixed random forest.

In [ ]:
# Direct NSPPK use with the best settings returned by .tune().
final_encoder = NSPPK(
    **best_params,
    nbits=NBITS,
    dense=False,
    parallel=PARALLEL,
)
feature_started = time.perf_counter()
X_full_train = final_encoder.fit_transform(full_train_graphs)
X_full_test = final_encoder.transform(full_test_graphs)
feature_seconds = time.perf_counter() - feature_started

final_classifier = RandomForestClassifier(**RF_SETTINGS)
fit_started = time.perf_counter()
final_classifier.fit(X_full_train, y_full_train)
classifier_fit_seconds = time.perf_counter() - fit_started
test_probabilities = final_classifier.predict_proba(X_full_test)[:, 1]
test_predictions = final_classifier.predict(X_full_test)

test_metrics, confusion, report_figure = plot_heldout_report(
    y_full_test,
    test_probabilities,
    test_predictions,
    best_params=best_params,
    nbits=NBITS,
    classifier_label='Random forest, 300 trees',
)
full_test_summary = pd.DataFrame([{
    'train_examples': len(y_full_train),
    'test_examples': len(y_full_test),
    'train_active': int(y_full_train.sum()),
    'test_active': int(y_full_test.sum()),
    'test_active_prevalence': float(y_full_test.mean()),
    'budget_per_class_requested': BUDGET_PER_CLASS,
    'budget_per_class_effective': effective_per_class,
    **best_params,
    'nbits': NBITS,
    **test_metrics,
    'feature_seconds': feature_seconds,
    'classifier_fit_seconds': classifier_fit_seconds,
}])
display(full_test_summary.round(4))
display(pd.DataFrame(confusion, index=['true_0', 'true_1'], columns=['pred_0', 'pred_1']))
REPORT_FIGURE = REPO_ROOT / 'data' / 'hiv_nsppk_bo_full_test_report.png'
report_figure.savefig(REPORT_FIGURE, dpi=180, bbox_inches='tight')
plt.show()
print('Saved test report figure:', REPORT_FIGURE)

## Save results

In [ ]:
RESULTS_PATH = REPO_ROOT / 'data' / 'hiv_nsppk_bo_results.csv'
bo_results.to_csv(RESULTS_PATH, index=False)
SUMMARY_PATH = REPO_ROOT / 'data' / 'hiv_nsppk_bo_full_test_metrics.csv'
full_test_summary.to_csv(SUMMARY_PATH, index=False)
print('Saved BO trials:', RESULTS_PATH)
print('Saved full test metrics:', SUMMARY_PATH)